# 3.2 Índice de Martonne → Base de Datos

Descarga los TIF del Índice de Martonne desde GEE (si no existen localmente),
muestrea el valor de categoría climática (1-6) en las coordenadas de cada parcela
y actualiza `parcela_inventario.martonneidx_id`.

**Criterio de asignación de período climático por inventario:**

Cada inventario se asocia al normal climático de 30 años más representativo de su época:
- IFN2 → TIF 1961-1991
- IFN3 → TIF 1972-2002
- IFN4 → TIF 1987-2017

**Clasificación Martonne (uint8, 0 = fuera de España):**
| Categoría | Rango | Descripción |
|-----------|-------|-------------|
| 1 | < 5 | Desierto |
| 2 | 5–10 | Semiárido |
| 3 | 10–20 | Mediterráneo |
| 4 | 20–30 | Húmedo |
| 5 | 30–60 | Perhúmedo |
| 6 | > 60 | Superhúmedo |

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
from sqlalchemy import text
from db import engine, DB_NAME

# ── Configuración ────────────────────────────────────────────────────────────
GEE_PROJECT  = os.environ['GEE_PROJECT']  # Google Cloud project with Earth Engine enabled
TIF_DIR      = 'datos_gee/martonne'
SCALE        = 1000   # resolución ERA5-Land en metros
SPAIN_BBOX   = [-10, 35, 4, 44]   # [west, south, east, north]

os.makedirs(TIF_DIR, exist_ok=True)

# Períodos climáticos → archivos TIF locales
PERIODS = {
    (1961, 1991): os.path.join(TIF_DIR, 'Clima_Martonne_Espana_GeoTiff_1961.0_1991.0.tif'),
    (1972, 2002): os.path.join(TIF_DIR, 'Clima_Martonne_Espana_GeoTiff_1972.0_2002.0.tif'),
    (1987, 2017): os.path.join(TIF_DIR, 'Clima_Martonne_Espana_GeoTiff_1987.0_2017.0.tif'),
}

# Normal climático asignado a cada inventario
INVENTORY_PERIOD = {
    2: (1961, 1991),
    3: (1972, 2002),
    4: (1987, 2017),
}

print('Configuración cargada.')
print(f'DB: {DB_NAME}')
print(f'TIF_DIR: {os.path.abspath(TIF_DIR)}')
for inv, period in INVENTORY_PERIOD.items():
    print(f'  IFN{inv} → {period[0]}-{period[1]}')

In [ ]:
import ee
import geemap

# ── Autenticación GEE ────────────────────────────────────────────────────────
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print('✓ GEE inicializado')

In [ ]:
def compute_martonne_image(start_year: int, end_year: int) -> 'ee.Image':
    """
    Calcula el índice de Martonne clasificado (categorías 1-6) para el período
    [start_year, end_year] usando ERA5-Land Monthly Aggregated.
    Devuelve una imagen uint8 clipeada a España.
    """
    era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
    years = ee.List.sequence(start_year, end_year)

    def annual_precip(year):
        year = ee.Number(year)
        return (era5
            .filterDate(ee.Date.fromYMD(year, 1, 1), ee.Date.fromYMD(year, 12, 31))
            .select('total_precipitation_sum')
            .sum()
            .multiply(1000)   # metros → mm
            .set('year', year))

    def annual_temp(year):
        year = ee.Number(year)
        return (era5
            .filterDate(ee.Date.fromYMD(year, 1, 1), ee.Date.fromYMD(year, 12, 31))
            .select('temperature_2m')
            .mean()
            .subtract(273.15)   # Kelvin → Celsius
            .set('year', year))

    mean_p = ee.ImageCollection(years.map(annual_precip)).mean().rename('P')
    mean_t = ee.ImageCollection(years.map(annual_temp)).mean().rename('T')

    martonne = mean_p.divide(mean_t.add(10))

    classified = (ee.Image(0)
        .where(martonne.lt(5),                              1)
        .where(martonne.gte(5).And(martonne.lt(10)),        2)
        .where(martonne.gte(10).And(martonne.lt(20)),       3)
        .where(martonne.gte(20).And(martonne.lt(30)),       4)
        .where(martonne.gte(30).And(martonne.lt(60)),       5)
        .where(martonne.gte(60),                            6)
        .rename('martonne_class')
        .toUint8())

    spain = (ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')
              .filter(ee.Filter.eq('country_na', 'Spain')))

    return classified.clip(spain).unmask(0)


def download_tif(start_year: int, end_year: int, output_path: str):
    """Descarga el TIF del período dado localmente. Sobreescribe si ya existe."""
    print(f'⏳ Descargando {start_year}-{end_year} → {output_path} ...')
    image = compute_martonne_image(start_year, end_year)
    region = ee.Geometry.Rectangle(SPAIN_BBOX)

    geemap.ee_export_image(
        image,
        filename=output_path,
        scale=SCALE,
        region=region,
        file_per_band=False,
    )
    print(f'✓ Guardado: {output_path}')


# ── Descargar los 3 TIFs (siempre, sin skip) ─────────────────────────────────
for (start, end), path in PERIODS.items():
    download_tif(start, end, path)

In [ ]:
# ── Añadir columna martonneidx_id a parcela_inventario (si no existe) ─────────
with engine.begin() as conn:
    cols = [r[0] for r in conn.execute(text('SHOW COLUMNS FROM parcela_inventario')).fetchall()]
    if 'martonneidx_id' not in cols:
        conn.execute(text('ALTER TABLE parcela_inventario ADD COLUMN martonneidx_id INT'))
        print('✓ Columna martonneidx_id añadida a parcela_inventario')
    else:
        print('✓ Columna martonneidx_id ya existe en parcela_inventario')

In [ ]:
# ── Cargar datos de la DB ─────────────────────────────────────────────────────
coords = pd.read_sql(
    'SELECT parcela_id, latitud AS latitude, longitud AS longitude FROM parcelas',
    engine
)
pi = pd.read_sql(
    'SELECT parcela_id, inventario_id FROM parcela_inventario',
    engine
)

df = pi.merge(coords, on='parcela_id', how='left')

n_before = len(df)
df = df.dropna(subset=['latitude', 'longitude'])
if len(df) < n_before:
    print(f'⚠ Descartadas {n_before - len(df)} filas sin coordenadas')

print(f'Total filas a procesar: {len(df)}')
print(df.groupby('inventario_id').size().rename('n_parcelas'))

In [ ]:
# ── Muestrear TIFs en las coordenadas de cada parcela ─────────────────────────
def sample_tif_values(tif_path: str, longitudes, latitudes) -> list:
    """
    Muestrea valores enteros de un TIF en EPSG:4326.
    Devuelve lista de enteros (0 = fuera del raster).
    """
    with rasterio.open(tif_path) as src:
        coords = list(zip(longitudes, latitudes))
        return [int(v[0]) for v in src.sample(coords)]


df['martonneidx_id'] = np.nan

for inv_id, period in INVENTORY_PERIOD.items():
    tif_path = PERIODS[period]
    idx = df[df['inventario_id'] == inv_id].index
    if len(idx) == 0:
        continue
    group = df.loc[idx]
    values = sample_tif_values(tif_path, group['longitude'].tolist(), group['latitude'].tolist())
    df.loc[idx, 'martonneidx_id'] = values
    print(f'  IFN{inv_id} ({period[0]}-{period[1]}): {len(idx)} parcelas → valores únicos: {sorted(set(values))}')

# 0 = fuera del raster → NULL
n_sin_dato = (df['martonneidx_id'] == 0).sum()
if n_sin_dato > 0:
    print(f'⚠ {n_sin_dato} parcelas con valor 0 (fuera del raster) → se guardarán como NULL')
    df.loc[df['martonneidx_id'] == 0, 'martonneidx_id'] = np.nan

df['martonneidx_id'] = df['martonneidx_id'].astype('Int64')

print(f'\nDistribución global de categorías Martonne:')
print(df['martonneidx_id'].value_counts().sort_index())

In [ ]:
# ── Actualizar parcela_inventario en bloque via tabla temporal ────────────────
# Preparar DataFrame con solo las columnas necesarias
df_update = df[['parcela_id', 'inventario_id', 'martonneidx_id']].copy()
df_update['inventario_id'] = df_update['inventario_id'].astype(int)

# Subir a tabla temporal
TEMP_TABLE = '_tmp_martonne'
with engine.begin() as conn:
    conn.execute(text(f'DROP TABLE IF EXISTS `{TEMP_TABLE}`'))

df_update.to_sql(
    TEMP_TABLE,
    engine,
    if_exists='replace',
    index=False,
    dtype={'martonneidx_id': None},
)

# UPDATE JOIN desde tabla temporal
with engine.begin() as conn:
    result = conn.execute(text(f"""
        UPDATE parcela_inventario pi
        JOIN `{TEMP_TABLE}` t
          ON pi.parcela_id    = t.parcela_id
         AND pi.inventario_id = t.inventario_id
        SET pi.martonneidx_id = t.martonneidx_id
    """))
    rows_updated = result.rowcount

    conn.execute(text(f'DROP TABLE IF EXISTS `{TEMP_TABLE}`'))

print(f'✓ Actualizadas {rows_updated} filas en parcela_inventario')

In [ ]:
# ── Verificación ──────────────────────────────────────────────────────────────
resumen = pd.read_sql("""
    SELECT
        inventario_id,
        martonneidx_id,
        COUNT(*) AS n
    FROM parcela_inventario
    GROUP BY inventario_id, martonneidx_id
    ORDER BY inventario_id, martonneidx_id
""", engine)

print('Distribución de martonneidx_id por inventario en la DB:')
print(resumen.to_string(index=False))

n_null = pd.read_sql(
    'SELECT COUNT(*) AS n FROM parcela_inventario WHERE martonneidx_id IS NULL',
    engine
)['n'][0]
total  = pd.read_sql('SELECT COUNT(*) AS n FROM parcela_inventario', engine)['n'][0]
print(f'\nNull: {n_null} / {total} ({100*n_null/total:.1f}%)')